***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [4. 可见度空间](4_0_introduction.ipynb)
    * 上一节：[4.5.2 改善 $uv$ 覆盖](4_5_2_uv_coverage_improving_your_coverage.ipynb)
    * 下一节：[4.x 延伸阅读与参考文献](4_x_further_reading_and_references.ipynb)

***


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


In [ ]:
HTML('../style/code_toggle.html')


## 4.6 Fourier 近似与 van Cittert-Zernike 定理

前面几个小节已经建立了三个事实：一条基线对应 $uv$ 平面中的一个采样点，相关器给出这个采样点处的复可见度，而地球自转和阵列布局决定这些采样点怎样分布。还剩下一个必须说清的问题：复可见度为什么能够和天空亮度建立 Fourier 关系？这个关系是否总是精确成立？

van Cittert-Zernike 定理给出的答案很优美，但它并不是一句孤立的口号。它把**空间非相干辐射源的亮度分布**与**接收平面上两点之间的互相干函数**联系起来。射电干涉仪把两面天线接收到的电压做互相关，测到的正是这种互相干性；当远场、窄带、小视场、方向响应可控等近似成立时，互相干函数就可以写成天空亮度的 Fourier 变换。

因此，本节的重点不是把定理背成一个公式，而是说明公式中每一项从哪里来、每一个近似删除了什么物理信息，以及在哪些观测情形下这些被删除的项会重新变得重要。


### 4.6.1 从互相干到可见度

考虑两面天线 $p$ 和 $q$，位置分别为 $\mathbf{r}_p$ 与 $\mathbf{r}_q$。从天空方向 $\mathbf{s}$ 到达的准单色电场，在两面天线处具有不同的几何延迟。若以 $\mathbf{b}_{pq}=\mathbf{r}_p-\mathbf{r}_q$ 表示基线向量，则几何光程差可写为

$$
\Delta L_{pq}(\mathbf{s})=\mathbf{b}_{pq}\cdot\mathbf{s},
$$

相应的相位差为 $2\pi \Delta L_{pq}/\lambda$。相关器计算的是两路复电压的时间平均：

$$
\Gamma_{pq}(\tau)=\left\langle E_p(t)E_q^*(t-\tau)\right\rangle .
$$

这里的尖括号表示在一个远长于电磁波周期、但短于天体和仪器显著变化时间的时间窗口内平均。这个平均非常关键：它抹去了快速振荡的载波，只留下稳定的相干信息，也就是可以被干涉仪测量的相位与幅度。

把天空看成许多方向元的叠加，天线 $p$ 接收到的电场可以形式化地写为

$$
E_p(t)=\int A_p(\mathbf{s})\,\epsilon(\mathbf{s},t-\tau_p)\,
    e^{-2\pi i\nu(t-\tau_p)}\,d\Omega,
$$

其中 $A_p(\mathbf{s})$ 是天线的方向响应，$\epsilon$ 描述天空在该方向上的随机辐射振幅，$\tau_p=\mathbf{r}_p\cdot\mathbf{s}/c$ 是几何延迟。把 $E_p$ 与 $E_q^*$ 相乘并取时间平均，会出现来自不同方向 $\mathbf{s}$ 和 $\mathbf{s}'$ 的交叉项。天文射电源通常可近似为热辐射或由大量独立辐射元组成的非相干辐射，因此不同方向之间的统计相关近似为零：

$$
\left\langle \epsilon(\mathbf{s},t)\epsilon^*(\mathbf{s}',t)\right\rangle
    =I_\nu(\mathbf{s})\,\delta(\mathbf{s}-\mathbf{s}').
$$

这个关系使双重积分坍缩成单个天空积分。于是零延迟附近的互相干函数可以写成

$$
\Gamma_{pq}\propto
\int A_p(\mathbf{s})A_q^*(\mathbf{s})I_\nu(\mathbf{s})
    e^{-2\pi i\,\mathbf{b}_{pq}\cdot\mathbf{s}/\lambda}\,d\Omega .
$$

这已经是 van Cittert-Zernike 定理的干涉测量形式：接收平面上两个点的互相干性，是天空亮度分布带有相位因子的积分。射电干涉仪中的复可见度 $V_{pq}$，正是经过增益、带通、延迟和归一化处理后的 $\Gamma_{pq}$。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(9, 5.6))
ax.set_aspect('equal')
ax.axis('off')

# Antennas and baseline
p = np.array([-1.3, -1.1])
q = np.array([1.2, -0.75])
s0 = np.array([0.0, 1.45])
s1 = np.array([1.15, 1.25])
origin = np.array([0.0, -0.95])

ax.scatter([p[0], q[0]], [p[1], q[1]], s=180, color='#31572c', zorder=3)
ax.text(p[0]-0.18, p[1]-0.35, 'p', fontsize=13)
ax.text(q[0]+0.08, q[1]-0.34, 'q', fontsize=13)
ax.annotate('', xy=q, xytext=p, arrowprops=dict(arrowstyle='<->', lw=2.0, color='#31572c'))
ax.text(-0.12, -1.18, r'baseline $\mathbf{b}_{pq}$', ha='center', fontsize=11, color='#31572c')

# Tangent plane
ax.plot([-1.8, 1.8], [1.05, 1.05], color='#555555', lw=1.2)
ax.text(-1.78, 1.12, 'tangent plane', fontsize=10, color='#555555')

for target, label, color in [(s0, r'phase center $\mathbf{s}_0$', '#7b2cbf'), (s1, r'direction $\mathbf{s}$', '#d65f00')]:
    ax.annotate('', xy=target, xytext=origin,
                arrowprops=dict(arrowstyle='->', lw=2.2, color=color))
    ax.scatter([target[0]], [target[1]], s=55, color=color, zorder=4)
    ax.text(target[0]+0.06, target[1]+0.08, label, fontsize=11, color=color)

# Direction-cosine offset
ax.annotate('', xy=(s1[0], 1.05), xytext=(s0[0], 1.05),
            arrowprops=dict(arrowstyle='<->', lw=1.6, color='#d65f00'))
ax.text(0.57, 0.86, r'offset $(l,m)$', ha='center', fontsize=11, color='#d65f00')

box = dict(boxstyle='round,pad=0.35', fc='white', ec='#444444', lw=1.0)
ax.text(0.0, -2.05,
        r'$\langle E_p E_q^*\rangle\;\longrightarrow\;V(u,v,w)$' + '\n' +
        r'$V=\int A I_\nu\,e^{-2\pi i\,\mathbf{b}\cdot\mathbf{s}/\lambda}\,d\Omega$',
        ha='center', va='center', fontsize=13, bbox=box)

ax.set_xlim(-2.3, 2.35)
ax.set_ylim(-2.45, 1.95)
fig.tight_layout()
fig.savefig(fig_dir / 'vcz_coherence_geometry.png', dpi=180, bbox_inches='tight')
plt.show()


### 4.6.2 相位中心、$(l,m,n)$ 与精确测量式

实际成像不会直接以整片天球为坐标，而是选定一个相位中心 $\mathbf{s}_0$，并在该方向建立局部正交基。天空中邻近相位中心的方向可写为

$$
\mathbf{s}=l\,\hat{\mathbf{e}}_l+m\,\hat{\mathbf{e}}_m+n\,\hat{\mathbf{e}}_n,
\qquad
n=\sqrt{1-l^2-m^2},
$$

其中 $l$ 和 $m$ 是相对于相位中心切平面的方向余弦，$n$ 描述该方向仍然位于单位天球上。把基线除以波长并投影到同一坐标系，可得

$$
\frac{\mathbf{b}_{pq}}{\lambda}=u\,\hat{\mathbf{e}}_l+v\,\hat{\mathbf{e}}_m+w\,\hat{\mathbf{e}}_n.
$$

相关器通常会对相位中心方向的几何延迟做补偿，因此相位因子中真正保留的是相对于相位中心的额外光程差：

$$
\frac{\mathbf{b}_{pq}\cdot(\mathbf{s}-\mathbf{s}_0)}{\lambda}
 = ul+vm+w(n-1).
$$

同时，球面面积元与方向余弦平面面积元之间满足

$$
d\Omega=\frac{dl\,dm}{n}.
$$

于是，在标量电场、窄带、远场、已完成相位跟踪的近似下，单个基线测到的可见度可写为

$$
\boxed{
V(u,v,w)=\int\!\!\int
\frac{A(l,m)I_\nu(l,m)}{n}
\exp\{-2\pi i[ul+vm+w(n-1)]\}\,dl\,dm
}
\tag{4.6.1}
$$

这个式子比常见的二维 Fourier 关系多出三类信息。第一，$A(l,m)$ 是主波束或更一般的方向响应，它限制视场并改变有效天空亮度。第二，$1/n$ 来自天球投影到切平面时的面积变换。第三，$w(n-1)$ 是球面几何留下的非二维 Fourier 相位项；当视场变宽、基线的 $w$ 分量较大时，它会成为宽场成像的主要误差来源之一。


### 4.6.3 二维 Fourier 近似从何而来

若视场足够小，则 $l^2+m^2\ll 1$，因此

$$
n=\sqrt{1-l^2-m^2}\simeq 1-\frac{l^2+m^2}{2}.
$$

在这个极限下，$1/n\simeq 1$。如果 $A(l,m)$ 在所考虑的小区域内变化缓慢，或者已经把主波束并入有效亮度 $I_\nu^{\rm app}(l,m)=A(l,m)I_\nu(l,m)$，则式 (4.6.1) 的主要困难只剩下 $w$ 项。该项带来的相位误差量级为

$$
|\Delta\phi_w|\simeq 2\pi |w|\frac{l^2+m^2}{2}
=\pi |w|\theta^2,
$$

其中 $\theta^2\simeq l^2+m^2$。当 $\pi |w|\theta^2\ll 1$，或者阵列几何使 $w=0$，式 (4.6.1) 退化为

$$
\boxed{
V(u,v)=\int\!\!\int I_\nu^{\rm app}(l,m)
\exp[-2\pi i(ul+vm)]\,dl\,dm
}
\tag{4.6.2}
$$

这就是综合孔径成像中最常用的二维 Fourier 近似。它说明：每一个可见度样本都是天空亮度在一个二维空间频率 $(u,v)$ 上的 Fourier 系数。短基线测量大角尺度结构，长基线测量细节；可见度相位记录结构相对于相位中心的位置偏移；不完整的 $uv$ 采样则导致脏束和旁瓣。

需要强调的是，式 (4.6.2) 并不否定式 (4.6.1)，而是式 (4.6.1) 在特定条件下的近似形式。许多实际观测正好处在二者之间：视场不算极小，$w$ 项不能完全忽略，主波束也有明显方向结构。后续宽场成像、$w$-projection、$w$-stacking、mosaicing 和 RIME 形式主义，都是围绕这些被近似掉的项展开。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

theta = np.linspace(0.0, 0.22, 400)
w_values = [10, 50, 200]

l = np.linspace(-0.35, 0.35, 2001)
dl = l[1] - l[0]
u = np.linspace(-45.0, 45.0, 700)
w_value = 18.0


def gaussian_1d(l_coords, sigma, center=0.0):
    profile = np.exp(-0.5 * ((l_coords - center) / sigma) ** 2)
    return profile / np.trapz(profile, l_coords)


def visibility_exact(u_coords, l_coords, brightness, w_term):
    n = np.sqrt(np.clip(1.0 - l_coords**2, 1e-12, None))
    phase = np.outer(u_coords, l_coords) + w_term * (n - 1.0)
    kernel = np.exp(-2j * np.pi * phase)
    return dl * (kernel @ (brightness / n))


def visibility_fourier_approx(u_coords, l_coords, brightness):
    kernel = np.exp(-2j * np.pi * np.outer(u_coords, l_coords))
    return dl * (kernel @ brightness)

models = [
    ('compact near center', gaussian_1d(l, sigma=0.025, center=0.015)),
    ('wide off center', gaussian_1d(l, sigma=0.095, center=0.16)),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for w in w_values:
    axes[0, 0].plot(theta, np.pi * w * theta**2, lw=2, label=fr'$w={w}$')
axes[0, 0].axhline(1.0, color='0.25', lw=1.2, ls='--')
axes[0, 0].set_xlabel(r'field angle $\theta$ [rad]')
axes[0, 0].set_ylabel(r'w-term phase $\pi |w|\theta^2$ [rad]')
axes[0, 0].set_title('w-term phase scale')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend(fontsize=9)

for ax, (label, sky) in zip(axes[0, 1:], models[:1]):
    ax.plot(l, sky / sky.max(), lw=2, color='#31572c')
    ax.set_title('sky slice: compact')
    ax.set_xlabel(r'$l$')
    ax.set_ylabel('normalized brightness')
    ax.grid(alpha=0.3)

for row, (label, sky) in enumerate(models):
    exact = visibility_exact(u, l, sky, w_value)
    approx_v = visibility_fourier_approx(u, l, sky)
    ax = axes[1, row]
    ax.plot(u, np.abs(exact), lw=2, label='exact kernel')
    ax.plot(u, np.abs(approx_v), lw=2, ls='--', label='2D Fourier')
    ax.set_title(label)
    ax.set_xlabel(r'$u$ [wavelengths]')
    ax.set_ylabel(r'$|V(u)|$')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)

fig.delaxes(axes[0, 1])
ax_sky = fig.add_subplot(2, 2, 2)
for label, sky in models:
    ax_sky.plot(l, sky / sky.max(), lw=2, label=label)
ax_sky.set_xlabel(r'$l$')
ax_sky.set_ylabel('normalized brightness')
ax_sky.set_title('test brightness distributions')
ax_sky.grid(alpha=0.3)
ax_sky.legend(fontsize=9)

fig.suptitle('When the 2D Fourier approximation begins to fail', y=0.99)
fig.tight_layout()
fig.savefig(fig_dir / 'vcz_fourier_approximation_limits.png', dpi=180, bbox_inches='tight')
plt.show()


### 4.6.4 近似条件的物理含义

上面的推导把若干假设压缩进了一个工作公式。为了在实际数据处理中使用这些公式，需要清楚每个假设的物理含义。

**空间非相干性。** 天空不同方向的辐射场在统计平均后互不相关，因此相关器输出可以按方向线性叠加。若观测目标本身具有空间相干辐射，或者近场目标无法用远方方向元描述，普通的 VCZ 推导就需要修改。

**远场条件。** 天源距离远大于阵列尺度，使到达阵列的波前可以用平面波近似。一个常用尺度是 Fraunhofer 距离 $R\gg B_{\max}^2/\lambda$。对天体射电源这一条件通常极好；对卫星、流星、空间碎片或电离层结构则未必成立。

**窄带与延迟跟踪。** 单个频率通道内，几何相位随频率的变化必须足够小。否则不同频率的条纹在平均时互相抵消，形成带宽去相关。现代相关器会先做延迟补偿，再把宽带数据分成许多窄频道；多频综合成像则进一步利用 $u=b/\lambda$ 随频率变化这一事实。

**方向响应可描述。** 主波束、馈源偏振响应、电离层和仪器方向误差都会乘到天空亮度上，或者在更完整的 RIME 中表现为方向相关 Jones 矩阵。本章暂时把它们收进 $A(l,m)$，但高动态范围和宽场成像不能忽略这些项。

**小视场或可处理的 $w$ 项。** 当 $\pi |w|\theta^2$ 很小，$w$ 项可忽略；当阵列共面且相位中心固定在阵列法线附近时，$w$ 项也可能自然很小。宽场低频阵列、长基线和低仰角观测往往不满足这个条件，需要使用三维成像或 $w$ 项校正算法。

**采样不完整。** VCZ 定理说明了理想连续可见度与天空亮度的关系，但真实阵列只在有限数量的 $uv$ 点上采样。若用采样函数 $S(u,v)$ 表示这一事实，则二维近似下的脏图为

$$
I_D(l,m)=\int\!\!\int S(u,v)V(u,v)e^{2\pi i(ul+vm)}\,du\,dv
      = I^{\rm app}(l,m)*B_D(l,m),
$$

其中 $B_D$ 是采样函数的 Fourier 变换，也就是脏束。因此，成像不是单纯“做一次反 Fourier 变换”，而是一个带有采样缺口、噪声、权重和仪器响应的反问题。


### 4.6.5 本章线索的收束

第四章从基线几何出发，逐步建立了射电干涉测量的核心语言。物理基线经过天球投影成为 $(u,v,w)$；相关器把两路电压的互相干性转换成复可见度；地球自转、阵列布局和频率覆盖决定可见度函数被采样的方式；van Cittert-Zernike 定理则说明，在明确近似条件下，这些可见度样本就是天空亮度分布的 Fourier 系数。

这条线索也自然引出后续两章。第五章将讨论如何从不完整的可见度采样形成图像，并解释脏图、脏束、加权和栅格化等概念；第六章进一步讨论去卷积，也就是如何在有限采样和噪声条件下恢复更接近真实天空的亮度模型。


***

* 下一节：[4.x 延伸阅读与参考文献](4_x_further_reading_and_references.ipynb)
